In [5]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

DB = Path("../data/db/euroleague.duckdb")
PREDICTIONS = Path("../data/derived/phase7/research/e2025_full_market_predictions.parquet")
con = duckdb.connect(str(DB), read_only=True)

market_summary = con.execute("""
SELECT count(*) AS row_count,
       count(DISTINCT matchday_number) AS matchday_count,
       min(matchday_number) AS min_matchday,
       max(matchday_number) AS max_matchday,
       min(observed_at) AS first_snapshot,
       max(observed_at) AS last_snapshot,
       count(DISTINCT fantasy_entity_id) AS entity_count,
       sum(CASE WHEN credits IS NULL THEN 1 ELSE 0 END) AS missing_credits,
       sum(CASE WHEN NOT credits_valid_pre_matchday THEN 1 ELSE 0 END) AS unsafe_credits
FROM leakage_safe_fantasy_market
WHERE season_code='E2025'
""").df()

pred_cols = con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [str(PREDICTIONS)]).df()
print(market_summary.to_string(index=False))
print()
print("Prediction columns:")
print(pred_cols[["column_name", "column_type"]].to_string(index=False))


 row_count  matchday_count  min_matchday  max_matchday                   first_snapshot                    last_snapshot  entity_count  missing_credits  unsafe_credits
     13191              38             1            38 2026-08-11 00:49:26.880487+03:00 2026-08-12 04:34:24.235615+03:00           365              0.0             0.0

Prediction columns:
                                       column_name              column_type
                                    player_game_id                  VARCHAR
                                            season                  VARCHAR
                                 season_start_year                  INTEGER
                                      round_number                  INTEGER
                                           game_id                  VARCHAR
                                         game_code                  INTEGER
                                  target_game_time TIMESTAMP WITH TIME ZONE
                                   

In [6]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import brier_score_loss, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

market = con.execute("""
SELECT season_code, matchday_number, player_id, fantasy_entity_id, fantasy_name,
       credits, observed_at
FROM leakage_safe_fantasy_market
WHERE season_code='E2025'
  AND credits_valid_pre_matchday
  AND credits IS NOT NULL
QUALIFY row_number() OVER (
    PARTITION BY season_code, matchday_number, fantasy_entity_id
    ORDER BY observed_at DESC, snapshot_record_id DESC
) = 1
""").df()

pred = con.execute("""
SELECT round_number AS matchday_number, player_id,
       max(name) AS name,
       avg(expected_fp) AS expected_fp,
       avg(baseline_expected_minutes) AS expected_minutes,
       avg(actual_fp) FILTER (WHERE played) AS actual_fp,
       avg(actual_minutes) FILTER (WHERE played) AS actual_minutes
FROM read_parquet(?)
WHERE season='E2025'
GROUP BY 1, 2
""", [str(PREDICTIONS)]).df()

frame = market.merge(pred, how="left", on=["matchday_number", "player_id"])
frame = frame.sort_values(["player_id", "matchday_number"]).reset_index(drop=True)
by_player = frame.groupby("player_id", dropna=False)
frame["previous_credits"] = by_player["credits"].shift(1)
frame["previous2_credits"] = by_player["credits"].shift(2)
frame["next_credits"] = by_player["credits"].shift(-1)
frame["next_matchday"] = by_player["matchday_number"].shift(-1)
frame["previous_actual_fp"] = by_player["actual_fp"].shift(1)
frame["previous_actual_minutes"] = by_player["actual_minutes"].shift(1)
frame["recent_actual_fp3"] = by_player["actual_fp"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).mean()
)
frame["credit_delta_1"] = frame["credits"] - frame["previous_credits"]
frame["credit_delta_2"] = frame["previous_credits"] - frame["previous2_credits"]
frame["target_delta"] = frame["next_credits"] - frame["credits"]
dataset = frame[
    (frame["next_matchday"] == frame["matchday_number"] + 1)
    & frame["player_id"].notna()
].copy()

features = [
    "credits", "credit_delta_1", "credit_delta_2", "expected_fp",
    "expected_minutes", "previous_actual_fp", "previous_actual_minutes",
    "recent_actual_fp3", "matchday_number",
]
reg_linear = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=10.0)),
])
reg_tree = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.05, max_iter=180, max_leaf_nodes=12,
        l2_regularization=2.0, random_state=8,
    )),
])
clf = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(C=0.25, max_iter=1000, random_state=8)),
])

folds = [(20, 21, 25), (25, 26, 30), (30, 31, 37)]
records = []
predicted_parts = []
for train_end, test_start, test_end in folds:
    train = dataset[dataset.matchday_number <= train_end]
    test = dataset[dataset.matchday_number.between(test_start, test_end)]
    if train.empty or test.empty:
        continue
    y_train = train.next_credits.to_numpy()
    y_test = test.next_credits.to_numpy()
    candidates = {
        "no_change": test.credits.to_numpy(),
        "mean_delta": test.credits.to_numpy() + train.target_delta.mean(),
        "recent_trend": test.credits.to_numpy() + test.credit_delta_1.fillna(0).to_numpy(),
    }
    for name, estimator in (("ridge", reg_linear), ("hist_gradient", reg_tree)):
        fitted = clone(estimator).fit(train[features], y_train)
        candidates[name] = fitted.predict(test[features])
    for name, values in candidates.items():
        records.append({
            "fold": f"MD{test_start}-{test_end}",
            "model": name,
            "n": len(test),
            "mae": mean_absolute_error(y_test, values),
            "rmse": mean_squared_error(y_test, values) ** 0.5,
        })
    inc_train = (train.target_delta > 0).astype(int)
    dec_train = (train.target_delta < 0).astype(int)
    inc_model = clone(clf).fit(train[features], inc_train)
    dec_model = clone(clf).fit(train[features], dec_train)
    predicted_parts.append(pd.DataFrame({
        "matchday_number": test.matchday_number.to_numpy(),
        "actual": y_test,
        "current": test.credits.to_numpy(),
        "p_increase": inc_model.predict_proba(test[features])[:, 1],
        "p_decrease": dec_model.predict_proba(test[features])[:, 1],
        "increase": (test.target_delta > 0).astype(int).to_numpy(),
        "decrease": (test.target_delta < 0).astype(int).to_numpy(),
    }))

metrics = pd.DataFrame(records)
pooled = metrics.groupby("model").apply(
    lambda g: pd.Series({
        "folds": len(g),
        "weighted_mae": np.average(g.mae, weights=g.n),
        "weighted_rmse": np.sqrt(np.average(g.rmse ** 2, weights=g.n)),
        "folds_beating_no_change_mae": int(sum(
            g.set_index("fold").mae
            < metrics[metrics.model == "no_change"].set_index("fold").mae
        )) if g.name != "no_change" else 0,
    }),
    include_groups=False,
).reset_index()
prob = pd.concat(predicted_parts, ignore_index=True)
prob_metrics = {
    "n": len(prob),
    "increase_rate": float(prob.increase.mean()),
    "decrease_rate": float(prob.decrease.mean()),
    "increase_brier": brier_score_loss(prob.increase, prob.p_increase),
    "decrease_brier": brier_score_loss(prob.decrease, prob.p_decrease),
}
print("Dataset:", len(dataset), "rows;", dataset.matchday_number.min(), "to", dataset.matchday_number.max())
print("Observed price changes:", dataset.target_delta.value_counts().sort_index().to_dict())
print()
print(metrics.round(4).to_string(index=False))
print()
print(pooled.round(4).to_string(index=False))
print()
print("Probability calibration:", {k: round(v, 4) for k, v in prob_metrics.items()})


Dataset: 11445 rows; 1 to 37
Observed price changes: {-0.9000000000000021: 1, -0.9000000000000004: 3, -0.8000000000000007: 9, -0.7999999999999989: 3, -0.7000000000000011: 12, -0.6999999999999993: 25, -0.6000000000000014: 21, -0.6000000000000005: 4, -0.5999999999999996: 74, -0.5000000000000009: 5, -0.5: 250, -0.4999999999999991: 4, -0.40000000000000213: 6, -0.40000000000000036: 333, -0.39999999999999947: 100, -0.3999999999999986: 64, -0.3000000000000007: 326, -0.2999999999999998: 370, -0.29999999999999893: 139, -0.29999999999999716: 2, -0.20000000000000284: 3, -0.20000000000000107: 153, -0.20000000000000018: 437, -0.1999999999999993: 356, -0.10000000000000142: 277, -0.10000000000000053: 759, -0.09999999999999964: 2123, -0.09999999999999787: 12, 0.0: 2824, 0.09999999999999787: 10, 0.09999999999999964: 541, 0.10000000000000053: 126, 0.10000000000000142: 117, 0.1999999999999993: 256, 0.20000000000000018: 195, 0.20000000000000107: 118, 0.20000000000000284: 6, 0.29999999999999716: 2, 0.29999

In [9]:
import sys
import importlib
sys.path.insert(0, str(Path("..").resolve()))
import src.control_center.price as price_module
price_module = importlib.reload(price_module)

price_audit = price_module.audit_and_train_price_model()
print("Acceptance:", price_audit.acceptance_status)
print("Training rows:", price_audit.training_rows)
print("No-change:", price_audit.validation["pooled"]["no_change"])
print("Ridge:", price_audit.validation["pooled"]["ridge"])
print("RMSE fold wins:", price_audit.validation["rmse_fold_wins"])
print("MAE fold wins:", price_audit.validation["mae_fold_wins"])
print("Leading inputs:", price_audit.leading_inputs)
print("Artifact:", price_audit.artifact_fingerprint, price_audit.artifact_path)


Acceptance: ACCEPTED_SECONDARY_RMSE_ONLY
Training rows: 11445
No-change: {'mae': 0.16379342032714575, 'rmse': 0.24553352520882013}
Ridge: {'mae': 0.1653894483583701, 'rmse': 0.24065183702820878}
RMSE fold wins: 3
MAE fold wins: 1
Leading inputs: ({'feature': 'credits', 'absolute_standardized_coefficient': 3.262928590697163}, {'feature': 'expected_fp', 'absolute_standardized_coefficient': 0.17159843743299938}, {'feature': 'expected_minutes', 'absolute_standardized_coefficient': 0.03696791116704982}, {'feature': 'previous_actual_fp', 'absolute_standardized_coefficient': 0.024377542756618135}, {'feature': 'credit_delta_1', 'absolute_standardized_coefficient': 0.022457863492827256})
Artifact: d7092c0edb06ffc79274e633d291fcc0ef374132a6b768acf0381020ce87dc49 /Users/heliasantoniou/PycharmProjects/ELfantasy/data/derived/phase8b/price_model/phase8b_e2025_next_price_ridge_v1.json


## Decision

The mechanism is statistically identifiable only weakly from safe pregame information.

- E2025 point-in-time rows: **11,445** across Matchdays 1–37.
- No-change: MAE **0.1638**, RMSE **0.2455**.
- Ridge: MAE **0.1654**, RMSE **0.2407**.
- Ridge improves RMSE in **3/3** chronological folds but MAE in only **1/3**.
- Increase/decrease probability Brier scores are **0.1619 / 0.2307**.
- Strongest standardized inputs: current credits, pregame expected FP, expected minutes, prior realized FP, and the latest credit delta.

Deployment status: **ACCEPTED_SECONDARY_RMSE_ONLY**, low confidence. The output is shown as separate price intelligence and is never used by the frozen Phase 7 Fantasy-score objective. No 2–3 Matchday forecast is released because this one-step signal is not strong enough to justify compounding.